# Exercise 3: prompt experiment – Flow and API

## Tasks

1. **One prompt:** write a prompt and create a picture in Flow.
2. **Vary:** change one building block of the prompt and create again.
3. **Repeat:** create several pictures for each version.
4. **Compare:** rate one feature of the pictures – by eye or with the API.

Images in Flow are free. Do not create videos yet.

## Setup

Select the **Python 3.11** kernel. The API key is only needed for the optional part at the end.

In [ ]:
# Standard Python libraries.
import os
from pathlib import Path

# Libraries installed from requirements.txt.
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

# Work inside this exercise folder, wherever Jupyter starts.
if not Path("data").exists():  # we are inside an exercise folder
    os.chdir("..")  # go to the main folder
os.chdir("03_Prompt_Experiment")

## 1. Write a prompt from building blocks

Describe what should be visible. Copy the printed prompt into Flow and create a picture.

In [ ]:
# The building blocks of the prompt.
blocks_a = {
    "subject": "a red wooden rowboat",
    "setting": "on a mountain lake",
    "light": "at dawn, soft light",
    "composition": "wide shot",
}

# Fill the template with the blocks.
TEMPLATE = "{subject}, {setting}, {light}, {composition}"
print("A:", TEMPLATE.format(**blocks_a))

## 2. Change one building block and create again

Only **one** block changes, the rest stays the same – so you know what caused the difference.

In [ ]:
# Copy the blocks and change only the composition.
blocks_b = blocks_a.copy()
blocks_b["composition"] = "wide shot, boat in the left third"
print("B:", TEMPLATE.format(**blocks_b))

# The feature you will rate on every picture.
FEATURE = "The center of the red rowboat is in the left third of the image."

## 3. Create three pictures per version and upload them

Use the same model and aspect ratio for all pictures. Upload them into the folder `flow_images/` next to this notebook and name them:

`variant_a_1`, `variant_a_2`, `variant_a_3`, `variant_b_1`, `variant_b_2`, `variant_b_3` (`.jpg` or `.png`)

In [ ]:
# Create the upload folder.
folder = Path("flow_images")
folder.mkdir(exist_ok=True)

# Find the six expected files and stop if one is missing or doubled.
names = ["variant_a_1", "variant_a_2", "variant_a_3", "variant_b_1", "variant_b_2", "variant_b_3"]
paths = []
for name in names:
    found = list(folder.glob(name + ".jpg")) + list(folder.glob(name + ".png"))
    assert len(found) == 1, f"flow_images/ needs exactly one {name}.jpg or {name}.png"
    paths.append(found[0])
print("All 6 images found.")

## 4. Compare the versions by eye

In [ ]:
# Show version A in the top row and version B in the bottom row.
fig, axes = plt.subplots(2, 3, figsize=(9, 4))
for i in range(6):
    ax = axes.flat[i]
    ax.imshow(Image.open(paths[i]))
    ax.set_title(names[i])
    ax.axis("off")
plt.show()

In [ ]:
# One row per picture.
ratings = pd.DataFrame({"name": names, "version": ["A", "A", "A", "B", "B", "B"]})

# Your rating per picture, in the order shown: True if the feature is visible.
# Replace these example values with your own.
ratings["me"] = [False, True, False, True, True, False]

# True counts as 1, False as 0 – so the mean is the share of "True".
ratings.groupby("version")["me"].mean()

## Optional: let the API rate the same feature

The same yes/no question for all six pictures. Does the API agree with you?

In [ ]:
# Libraries for the API part.
import base64
import json
import mimetypes

from openai import OpenAI
from pydantic import BaseModel

# Connect to OpenAI. The key is read from OPENAI_API_KEY; busy requests are retried.
client = OpenAI(max_retries=5, timeout=60)
MODEL = "gpt-6-luna"

## Helper functions for the API calls

- `image_to_input(path)` turns an image file into text (base64), because the API request is sent as text.
- `ask(prompt, answer_format, images)` sends the prompt and the images, writes a log line to `api_log.jsonl` and returns the answer as a Python dict.

In [ ]:
# Function 1: turn an image file into an API input.
def image_to_input(path):
    image_type = mimetypes.guess_type(path)[0]  # e.g. "image/jpeg"
    image_text = base64.b64encode(path.read_bytes()).decode()  # bytes -> text
    return {
        "type": "input_image",
        "image_url": f"data:{image_type};base64,{image_text}",
        "detail": "low",  # low detail = fewer tokens = cheaper
    }


# Function 2: send a prompt and images, get the answer as a dict.
def ask(prompt, answer_format, images):
    # 1. Put the text and the images into one message.
    content = [{"type": "input_text", "text": prompt}]
    for path in images:
        content.append(image_to_input(path))

    # 2. Send the message. The answer must follow answer_format.
    response = client.responses.parse(
        model=MODEL,
        input=[{"role": "user", "content": content}],
        text_format=answer_format,
        reasoning={"effort": "none"},
        max_output_tokens=500,
        store=False,
    )

    # 3. Log prompt, model and raw answer.
    entry = {"prompt": prompt, "model": MODEL, "answer": response.output_text}
    with open("api_log.jsonl", "a", encoding="utf-8") as log:
        log.write(json.dumps(entry, ensure_ascii=False) + "\n")

    # 4. Stop with a clear message if there is no valid answer.
    if response.output_parsed is None:
        raise ValueError(f"No valid answer for {images}. Look at api_log.jsonl.")

    # 5. Return the answer as a Python dict.
    return response.output_parsed.model_dump()

In [ ]:
# The answer format: yes or no, plus a short reason.
class Rating(BaseModel):
    matches: bool
    reason: str


# The same question for every picture.
PROMPT = f"Is this true? {FEATURE} If there is no boat, answer false. Give a short reason."

In [ ]:
# Ask the model about every picture.
api = []
for path in paths:
    api.append(ask(PROMPT, Rating, [path])["matches"])

# Your rating and the API rating side by side.
ratings["api"] = api
display(ratings)
ratings.groupby("version")[["me", "api"]].mean()

**Check yourself:** Why does the same prompt give different pictures – and what does that mean for your experiment?

### Jupyter notebook --footer info--

In [ ]:
# Print system and version info for this run.
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')